# 👁️ Panopti: Pre-Execution Guardrails & Invariant Engine Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shinraya-Tech-Labs/AgentWatch/blob/main/examples/quickstart_panopti_guardrails.ipynb)
[![License](https://img.shields.io/badge/license-Apache_2.0-blue?style=flat-square)](https://github.com/Shinraya-Tech-Labs/AgentWatch/blob/main/LICENSE)

### Overview
Autonomous AI agents are prone to silent reasoning failures, prompt injection exploits, hallucinated arguments, and unintended side effects. Traditional LLM observability platforms operate **post-mortem**—tracing what went wrong only *after* a database has been dropped or private credentials exfiltrated.

**Panopti** (formerly AgentWatch) acts as an **active, pre-execution runtime hypervisor**:
- Evaluates tool intent against deterministic temporal and safety invariants.
- Intercepts and vetoes rogue operations *before* they reach production drivers.
- Produces cryptographic Merkle witness audit proofs for verifiable regulatory compliance (e.g. EU AI Act Article 15).

---
### In this Quickstart, you will:
1. **Install Panopti / AgentWatch**.
2. **Configure Pre-Execution Invariants** (`Disallows`, `RequiresPrior`).
3. **Run a Nominal Tool Call** (Validated & Allowed ✅).
4. **Simulate a Rogue Invariant Breach** (Intercepted & Vetoed 🛑).
5. **Inspect the Cryptographic Witness Proof & Attestation Seal** (Tamper-evident 🔏).

## Step 1: Install Panopti (`agentwatch-ai`)

Panopti is published to PyPI as `agentwatch-ai` (with `panopti-ai` alias).

In [ ]:
# Cell 1: Install Panopti runtime guard
!pip install -q agentwatch-ai

## Step 2: Configure Pre-Execution Guard & Invariant Rules

Panopti enforces safety through **Contracts** evaluated against the agent's historical execution stream before any tool executes:
- `Disallows`: Explicitly prohibits specific destructive tools or operations.
- `RequiresPrior`: Mandates prerequisite safety checkpoints (e.g., an agent *must* pass identity validation before executing financial transactions).

Let's configure `PanoptiGuard` with these invariants.

In [ ]:
# Cell 2: Setting up PreExecutionGuard with invariant rules
try:
    from panopti import (
        PanoptiGuard,
        Disallows,
        RequiresPrior,
        WitnessChain,
        AttestationEngine,
    )
except ImportError:
    from panopti.guard import PanoptiGuard
    from panopti.contracts import Disallows, RequiresPrior
    from panopti.court import WitnessChain
    from panopti.attest import AttestationEngine

# Initialize the active pre-execution guard
guard = PanoptiGuard(critical_confidence_floor=0.40)

# Invariant 1: Strictly prohibit destructive schema operations
guard.register_contract(
    Disallows(
        target_tool="drop_table",
        reason="Direct table drops and schema modifications are strictly banned in production.",
    )
)

# Invariant 2: Sequential prerequisite contract
# The agent cannot call execute_settlement unless verify_credentials succeeded within the last 5 steps
guard.register_contract(
    RequiresPrior(
        target_tool="execute_settlement",
        required_prior="verify_credentials",
        within_steps=5,
        custom_message="Security Invariant Breach: 'execute_settlement' requires prior successful verification ('verify_credentials').",
    )
)

print("🛡️ Panopti PreExecutionGuard initialized with active invariant contracts!")

## Step 3: Running a Nominal Agent Tool Call (Allowed ✅)

Here an autonomous agent proposes a valid workflow:
1. First, it validates credentials via `verify_credentials`.
2. Then, it proceeds with `execute_settlement`.

Panopti evaluates each intent pre-flight, verifies compliance with all active contracts, and grants permission.

In [ ]:
# Cell 3: Running nominal agent tool calls (Allowed)

# Step A: Agent performs prerequisite verification
print("▶ [Step 1] Proposed Action: verify_credentials")
decision_1 = guard.check(
    tool_name="verify_credentials",
    arguments={"account_id": "ACC-4902", "auth_method": "mfa_token"},
    confidence=0.98,
)

print(f"  • Panopti Verdict: Allowed={decision_1.allowed} | Risk={decision_1.risk_level}")
if decision_1.allowed:
    # Tool executes safely and records success in the Panopti execution history
    guard.record_execution("verify_credentials", {"account_id": "ACC-4902"})
    print("  • Execution: SUCCESS -> Identity token verified.\n")

# Step B: Agent executes settlement (allowed because verify_credentials succeeded)
print("▶ [Step 2] Proposed Action: execute_settlement")
decision_2 = guard.check(
    tool_name="execute_settlement",
    arguments={"account_id": "ACC-4902", "amount_usd": 1250.00},
    confidence=0.95,
)

print(f"  • Panopti Verdict: Allowed={decision_2.allowed} | Risk={decision_2.risk_level}")
if decision_2.allowed:
    guard.record_execution("execute_settlement", {"amount_usd": 1250.00})
    print("  • Execution: SUCCESS -> Settlement completed within safety boundaries!")

## Step 4: Intercepting & Vetoing Rogue Tool Calls (Intercepted 🛑)

Now let's simulate two real-world failure modes:
1. **Direct Destructive Call**: An adversarial prompt injection instructs the agent to run `drop_table` (`Disallows` breach).
2. **Authentication Bypass**: An ungrounded reasoning loop tries to call `execute_settlement` without running `verify_credentials` (`RequiresPrior` breach).

Panopti intercepts both calls **before execution**, protecting backend services.

In [ ]:
# Cell 4: Running rogue tool calls (Intercepted & Vetoed)

print("--- Test 1: Adversarial Prompt Injection (DROP TABLE) ---")
rogue_decision_1 = guard.check(
    tool_name="drop_table",
    arguments={"table": "users", "cascade": True},
    confidence=0.88,
)

print(f"  • Pre-Execution Intercept: VETOED (Blocked={rogue_decision_1.blocked})")
print(f"  • Risk Level            : {rogue_decision_1.risk_level} (Score: {rogue_decision_1.risk_score})")
print(f"  • Policy Veto Reason    : {rogue_decision_1.reason}\n")

print("--- Test 2: Unverified Action Bypass (Missing Prerequisite) ---")
# Initialize a fresh guard with no prior verification history
isolated_guard = PanoptiGuard()
isolated_guard.register_contract(
    RequiresPrior(
        target_tool="execute_settlement",
        required_prior="verify_credentials",
    )
)

rogue_decision_2 = isolated_guard.check(
    tool_name="execute_settlement",
    arguments={"account_id": "UNKNOWN", "amount_usd": 5000000.00},
    confidence=0.92,
)

print(f"  • Pre-Execution Intercept: VETOED (Blocked={rogue_decision_2.blocked})")
print(f"  • Invariant Breach      : {rogue_decision_2.reason}")
print("\n🛡️ ZERO DATA LOSS: All unauthorized actions halted prior to driver execution.")

## Step 5: Cryptographic Merkle Witness & Attestation Seal (🔏)

For regulatory auditability (e.g., EU AI Act, SOC2, HIPAA), every decision must be provably tamper-evident.

Panopti seals execution events using:
1. **`WitnessChain`**: A SHA-256 cryptographic hash chain where each event encapsulates the hash of its parent.
2. **`AttestationEngine`**: Generates an HMAC-SHA256 digital certificate proving whether the trajectory was compliant or experienced invariant vetoes.

In [ ]:
# Cell 5: Inspecting cryptographic Merkle witness & attestation seal

# 1. Initialize cryptographic witness court for this agent run
witness_court = WitnessChain(session_id="colab_quickstart_session_01")

# 2. Record nominal and intercepted entries into the immutable ledger
entry_1 = witness_court.record(
    actor="Agent-Alpha",
    action="verify_credentials",
    payload={"verdict": "ALLOW", "account_id": "ACC-4902"},
)

entry_2 = witness_court.record(
    actor="Agent-Alpha",
    action="drop_table",
    payload={"verdict": "VETO", "reason": rogue_decision_1.reason},
)

print(f"Witness Chain Entries: {witness_court.length}")
print(f"Cryptographic Root Hash: {witness_court.root_hash}")

# 3. Export and mathematically verify the chain integrity
proof = witness_court.export_proof()
is_chain_valid, err = WitnessChain.verify_proof(proof)
print(f"Merkle Chain Mathematical Integrity: {is_chain_valid} (Error: {err})\n")

# 4. Generate an immutable Attestation Seal
seal = AttestationEngine.create_seal(
    merkle_root=witness_court.root_hash,
    tool_count=witness_court.length,
    confidence_floor=0.40,
    is_compliant=False,  # Marked False because a veto was enforced
    loop_detected=False,
    secret_key="panopti-quickstart-audit-key",
)

print("=== 📜 Cryptographic Attestation Certificate ===")
print(seal.to_json())

# 5. Cryptographically verify signature
is_authentic = AttestationEngine.verify_seal(seal, secret_key="panopti-quickstart-audit-key")
print(f"\nHMAC-SHA256 Signature Verification: {is_authentic} (TAMPER-PROOF AUDIT CONFIRMED) ✅")

---
### Summary & Next Steps

You have successfully:
- Configured Panopti pre-execution invariant rules (`Disallows`, `RequiresPrior`).
- Evaluated safe tool executions and recorded them in the execution history.
- Intercepted and blocked rogue SQL schema operations and unauthorized prerequisite bypasses.
- Generated and mathematically verified a tamper-evident cryptographic Attestation Seal.

**Explore Further:**
- 📚 [Documentation & Architecture](https://github.com/Shinraya-Tech-Labs/AgentWatch/tree/main/docs)
- 💬 [Join Community Discord](https://discord.gg/UT9uaeY46e)
- 🌟 [Star AgentWatch on GitHub](https://github.com/Shinraya-Tech-Labs/AgentWatch)